# Task 1: Diagnostic Enhancement of Chest X-Rays
Student: Sabeeh | Roll number: 23K-0002

## 1. Imports and configuration
All paths below are relative to this task folder. Place one dataset image at `data/sample_xray.png`.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

TASK_DIR = Path.cwd()
if TASK_DIR.name != "Task_1_Chest_XRay" and (TASK_DIR / "Task_1_Chest_XRay").exists():
    TASK_DIR = TASK_DIR / "Task_1_Chest_XRay"
DATA_DIR = TASK_DIR / "data"
OUTPUT_DIR = TASK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_PATH = next((DATA_DIR / f"sample_xray{ext}" for ext in (".png", ".jpg", ".jpeg") if (DATA_DIR / f"sample_xray{ext}").is_file()), DATA_DIR / "sample_xray.png")
print(f"Sample path: {IMAGE_PATH}")

## 2. Load data and view original
The notebook accepts a PNG/JPG/JPEG sample placed at the configured relative path.

In [ ]:
if not IMAGE_PATH.is_file():
    raise FileNotFoundError("Place one chest X-ray at Task_1_Chest_XRay/data/sample_xray.png (or .jpg/.jpeg). Download only a single sample from the dataset linked in README.md.")
raw = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
if raw is None:
    raise ValueError(f"OpenCV could not read {IMAGE_PATH}; use a valid PNG/JPG/JPEG image.")
cv2.imwrite(str(OUTPUT_DIR / "original_xray.png"), raw)
plt.figure(figsize=(6, 6)); plt.imshow(raw, cmap="gray"); plt.title("Original grayscale X-ray"); plt.axis("off"); plt.show()

## 3. Histogram equalization
For grayscale data, OpenCV redistributes the cumulative intensity histogram across the available range.

In [ ]:
equalized = cv2.equalizeHist(raw)
cv2.imwrite(str(OUTPUT_DIR / "histogram_equalized.png"), equalized)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(raw, cmap="gray"); axes[0].set_title("Raw")
axes[1].imshow(equalized, cmap="gray"); axes[1].set_title("Histogram equalized")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. False color and mathematical color balance
JET is applied to the equalized uint8 grayscale image. Color balancing subtracts each channel mean deviation from neutral gray and clips to the valid range.

In [ ]:
jet = cv2.applyColorMap(equalized, cv2.COLORMAP_JET)

jet_f = jet.astype(np.float32)
channel_means = jet_f.mean(axis=(0, 1), keepdims=True)
neutral_mean = channel_means.mean()
balanced = np.clip(jet_f + (neutral_mean - channel_means), 0, 255).astype(np.uint8)
cv2.imwrite(str(OUTPUT_DIR / "jet_heatmap.png"), jet)
cv2.imwrite(str(OUTPUT_DIR / "color_balanced.png"), balanced)
plt.figure(figsize=(11, 5))
plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)); plt.title("JET heatmap"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(balanced, cv2.COLOR_BGR2RGB)); plt.title("Color balanced"); plt.axis("off")
plt.tight_layout(); plt.show()

## 5. Strict high-intensity threshold
Only pixels strictly greater than the configured grayscale threshold are retained; all other pixels become black.

In [ ]:
DENSE_THRESHOLD = 200  
thresholded = np.where(raw > DENSE_THRESHOLD, raw, 0).astype(np.uint8)
cv2.imwrite(str(OUTPUT_DIR / "thresholded.png"), thresholded)
plt.figure(figsize=(6,6)); plt.imshow(thresholded, cmap="gray"); plt.title(f"Pixels with intensity > {DENSE_THRESHOLD}"); plt.axis("off"); plt.show()

## 6. Logarithmic transform on raw underexposed image
Normalize to [0,1], apply `log(1+x) / log(2)`, then scale and round to uint8. This expands the relative visibility of darker values.

In [ ]:
raw01 = raw.astype(np.float32) / 255.0
log_image = np.clip(np.log1p(raw01) / np.log(2.0), 0.0, 1.0)
log_image = np.round(log_image * 255).astype(np.uint8)
cv2.imwrite(str(OUTPUT_DIR / "logarithmic.png"), log_image)

## 7. Power-law / gamma correction
With gamma below one, normalized values are raised to gamma, brightening darker values.

In [ ]:
GAMMA = 0.65
if not 0 < GAMMA < 1: raise ValueError("GAMMA must be less than 1.0 and greater than zero")
gamma_image = np.round(np.power(raw01, GAMMA) * 255).clip(0, 255).astype(np.uint8)
cv2.imwrite(str(OUTPUT_DIR / "gamma_corrected.png"), gamma_image)

## 8. Save comparison and conclude

In [ ]:
images = [("Original",raw),("Equalized",equalized),("Thresholded",thresholded),("Log",log_image),("Gamma",gamma_image)]
fig, axes = plt.subplots(1, len(images), figsize=(16,4))
for ax,(title,img) in zip(axes,images): ax.imshow(img,cmap="gray",vmin=0,vmax=255); ax.set_title(title); ax.axis("off")
plt.tight_layout(); fig.savefig(OUTPUT_DIR / "comparison.png", dpi=150, bbox_inches="tight"); plt.show()
print("Saved outputs to", OUTPUT_DIR.resolve())